In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D


# ---------------------------------------------------------------- style
MODELS = ["DPOT", "DISCO", "HFluxNO"]  # legend and draw order
# colorcet glasbey_bw indices 41, 85, 68 (same as Fig. 3)
COLORS = {"HFluxNO": "#3715ff", "DPOT": "#ff2747", "DISCO": "#00653c"}
MARKERS = {"HFluxNO": "o", "DPOT": "s", "DISCO": "^"}
OFFSETS = {"DPOT": -0.24, "DISCO": 0.0, "HFluxNO": 0.24}  # within-group x offsets
COND_DX = 0.065  # lower / upper condition drawn at xm -/+ COND_DX
FILL_STYLE = "three"  # "three": filled / half / hollow;  "two": filled / hollow


def _darken(hexc: str, f: float = 0.6) -> tuple:
    r, g, b = mpl.colors.to_rgb(hexc)
    return (r * f, g * f, b * f)


def _fill(cond: str, fam: str):
    """Return (mfc, fillstyle) for a condition."""
    if cond == "ID":
        return "full"
    if cond == "shock IC":
        return "none"
    # GRF IC on the unseen flux
    return "left" if FILL_STYLE == "three" else "full"


# (axis label, family key, lower condition, upper condition)
# "ID" is drawn filled; anything else hollow.
GROUPS = [
    ("1D cubic", "1D cubic", "ID", "shock IC"),
    ("Shallow\nwater", "1D shallow water", "ID", "shock IC"),
    ("Viscous\nBurgers", "1D viscous Burgers", "ID", "shock IC"),
    ("2D cubic", "2D cubic", "ID", "shock IC"),
    ("2D Euler", "2D Euler", "ID", "shock IC"),
    ("Sine flux\n(unseen)", "Sine flux", "GRF IC", "shock IC"),
]
GAP_BEFORE = {"Sine flux"}  # extra horizontal gap + divider before these

METRICS = [("l2", r"Rel. $\ell^2$"), ("linf", r"Rel. $\ell^\infty$")]

RC = {
    "font.family": "serif",
    "font.size": 8,
    "axes.labelsize": 8,
    "axes.titlesize": 8,
    "xtick.labelsize": 7.5,
    "ytick.labelsize": 7.5,
    "legend.fontsize": 7.5,
    "axes.linewidth": 0.6,
    "xtick.major.width": 0.5,
    "ytick.major.width": 0.5,
    "ytick.minor.width": 0.4,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
}


def _positions():
    """x coordinate of each group, with a wider gap before flagged groups."""
    xs, x = [], 0.0
    for _, fam, _, _ in GROUPS:
        if fam in GAP_BEFORE and xs:
            x += 0.6
        xs.append(x)
        x += 1.0
    return xs


def plot_accuracy(
    results: dict, out: str | None = None, width: float = 5.5, height: float = 2.9
):
    """Draw the figure and optionally save it (any matplotlib format)."""
    xs = _positions()
    with mpl.rc_context(RC):
        fig, axes = plt.subplots(
            2, 1, figsize=(width, height), sharex=True, gridspec_kw={"hspace": 0.12}
        )
        for ax, (metric, ylabel) in zip(axes, METRICS):
            ax.set_yscale("log")
            vals = [
                v[metric][0]
                for fam in results.values()
                for cond in fam.values()
                for v in cond.values()
                if metric in v
            ]
            if vals:
                ax.set_ylim(min(vals) / 1.6, max(vals) * 1.6)
            ax.set_ylabel(ylabel)
            ax.yaxis.grid(True, which="major", color="0.88", lw=0.5)
            ax.set_axisbelow(True)
            for spine in ("top", "right"):
                ax.spines[spine].set_visible(False)
            ax.tick_params(axis="x", length=0)

            for x, (_, fam, lo, hi) in zip(xs, GROUPS):
                fam_res = results.get(fam, {})
                drawn = False
                for model in MODELS:
                    xm = x + OFFSETS[model]
                    c = COLORS[model]
                    pts = {}
                    for cond in (lo, hi):
                        v = fam_res.get(cond, {}).get(model, {}).get(metric)
                        if v is not None:
                            pts[cond] = v
                    xpos = {lo: xm - COND_DX, hi: xm + COND_DX}
                    for cond, (m, sd) in pts.items():
                        fs = _fill(cond, fam)
                        # whisker behind the marker, in a darker shade of the
                        # model colour; a whisker longer than the marker radius
                        # shows on both sides, a shorter one is hidden (std
                        # below ~5% of the mean, see caption)
                        ax.errorbar(
                            xpos[cond],
                            m,
                            yerr=sd,
                            fmt="none",
                            ecolor=_darken(c),
                            elinewidth=0.7,
                            capsize=1.5,
                            capthick=0.7,
                            zorder=3,
                        )
                        ax.plot(
                            xpos[cond],
                            m,
                            marker=MARKERS[model],
                            ms=5.5,
                            mfc=c if fs != "none" else "white",
                            mfcalt="white",
                            fillstyle=fs if fs != "none" else "full",
                            mec=c,
                            mew=0.9,
                            ls="none",
                            zorder=4,
                        )
                        drawn = True
                if not drawn and ax is axes[0]:
                    ax.text(
                        x,
                        0.5,
                        "pending",
                        ha="center",
                        va="center",
                        fontsize=7,
                        color="0.5",
                        style="italic",
                        transform=ax.get_xaxis_transform(),
                    )

            # divider before the unseen-flux group
            for x, (_, fam, _, _) in zip(xs, GROUPS):
                if fam in GAP_BEFORE:
                    ax.axvline(x - 0.8, color="0.75", lw=0.6, ls=(0, (3, 3)), zorder=1)

        axes[-1].set_xticks(xs)
        axes[-1].set_xticklabels([g[0] for g in GROUPS])
        axes[-1].set_xlim(xs[0] - 0.6, xs[-1] + 0.6)

        # legend: model identity (colour + shape) and condition (fill)
        handles = [
            Line2D(
                [],
                [],
                marker=MARKERS[m],
                color=COLORS[m],
                mfc=COLORS[m],
                ls="none",
                ms=5,
                label=m,
            )
            for m in ["HFluxNO", "DPOT", "DISCO"]
        ]
        handles += [
            Line2D(
                [],
                [],
                marker="o",
                color="0.3",
                mfc="0.3",
                ls="none",
                ms=5,
                label="in distribution",
            ),
        ]
        if FILL_STYLE == "three":
            handles.append(
                Line2D(
                    [],
                    [],
                    marker="o",
                    color="0.3",
                    mfc="0.3",
                    mfcalt="white",
                    fillstyle="left",
                    ls="none",
                    ms=5.5,
                    label="unseen flux, GRF IC",
                )
            )
        handles.append(
            Line2D(
                [],
                [],
                marker="o",
                color="0.3",
                mfc="white",
                ls="none",
                ms=5,
                label="shock IC",
            )
        )
        axes[0].legend(
            handles=handles,
            ncol=len(handles),
            loc="lower center",
            bbox_to_anchor=(0.5, 1.0),
            frameon=False,
            handletextpad=0.4,
            columnspacing=1.2,
        )

        if out:
            fig.savefig(out, bbox_inches="tight", pad_inches=0.02, dpi=600)
        return fig

In [ ]:
results = {
    "1D cubic": {
        "ID": {  # Table 1
            "DPOT": {"l2": (1.36e-1, 3.12e-3), "linf": (6.78e-1, 9.65e-3)},
            "DISCO": {"l2": (8.22e-2, 5.58e-3), "linf": (4.55e-1, 3.12e-2)},
            "HFluxNO": {"l2": (5.21e-2, 2.15e-3), "linf": (3.68e-1, 1.56e-2)},
        },
        "shock IC": {  # Table 2, "1D cubic" block
            "DPOT": {"l2": (1.58e-1, 3.95e-3), "linf": (8.23e-1, 3.82e-2)},
            "DISCO": {"l2": (8.58e-2, 5.69e-3), "linf": (4.73e-1, 2.63e-2)},
            "HFluxNO": {"l2": (6.68e-2, 1.69e-3), "linf": (4.35e-1, 8.96e-3)},
        },
    },
    "1D shallow water": {
        "ID": {
            "DPOT": {"l2": (1.05e-1, 3.66e-3), "linf": (4.63e-1, 1.00e-2)},
            "DISCO": {"l2": (1.11e-1, 5.82e-3), "linf": (4.45e-1, 2.01e-2)},
            "HFluxNO": {"l2": (7.55e-2, 1.26e-3), "linf": (3.98e-1, 4.21e-3)},
        },
        "shock IC": {
            "DPOT": {"l2": (4.59e-1, 6.07e-3), "linf": (9.77e-1, 5.47e-3)},
            "DISCO": {"l2": (3.35e-1, 1.01e-2), "linf": (9.40e-1, 6.75e-2)},
            "HFluxNO": {"l2": (4.28e-1, 6.62e-3), "linf": (8.47e-1, 1.28e-2)},
        },
    },
    "1D viscous Burgers": {
        "ID": {
            "DPOT": {"l2": (4.66e-2, 1.07e-3), "linf": (1.56e-1, 4.39e-3)},
            "DISCO": {"l2": (6.56e-2, 3.35e-2), "linf": (3.43e-1, 2.34e-1)},
            "HFluxNO": {"l2": (1.72e-2, 2.35e-4), "linf": (6.46e-2, 6.51e-4)},
        },
        "shock IC": {
            "DPOT": {"l2": (4.50e-2, 3.19e-3), "linf": (1.78e-1, 1.17e-2)},
            "DISCO": {"l2": (4.53e-2, 1.41e-2), "linf": (2.41e-1, 1.25e-1)},
            "HFluxNO": {"l2": (2.21e-2, 5.04e-4), "linf": (9.99e-2, 9.47e-4)},
        },
    },
    "2D cubic": {
        "ID": {
            "DPOT": {"l2": (2.19e-1, 1.49e-2), "linf": (7.84e-1, 4.33e-2)},
            "DISCO": {"l2": (3.68e-2, 4.29e-3), "linf": (1.40e-1, 1.11e-2)},
            "HFluxNO": {"l2": (1.31e-2, 6.48e-4), "linf": (6.38e-2, 3.44e-3)},
        },
        "shock IC": {
            "DPOT": {"l2": (8.12e-1, 2.64e-2), "linf": (1.69e0, 1.16e-1)},
            "DISCO": {"l2": (2.04e-1, 1.14e-2), "linf": (6.79e-1, 3.47e-2)},
            "HFluxNO": {"l2": (1.76e-1, 2.43e-2), "linf": (5.65e-1, 7.46e-2)},
        },
    },
    "2D Euler": {  # 20-step rollout; single-step numbers in full_results_tables
        "ID": {
            "DPOT": {"l2": (2.23e-1, 7.97e-3), "linf": (7.13e-1, 2.14e-2)},
            "DISCO": {"l2": (1.27e-1, 1.12e-2), "linf": (5.45e-1, 3.82e-2)},
            "HFluxNO": {"l2": (3.94e-2, 8.40e-4), "linf": (3.27e-1, 6.30e-3)},
        },
        "shock IC": {
            "DPOT": {"l2": (1.77e-1, 6.98e-3), "linf": (7.81e-1, 4.43e-2)},
            "DISCO": {"l2": (1.05e-1, 1.27e-2), "linf": (4.97e-1, 6.11e-2)},
            "HFluxNO": {"l2": (2.68e-2, 3.72e-4), "linf": (2.37e-1, 3.51e-3)},
        },
    },
    "Sine flux": {  # cubic-trained models on the unseen sine flux (Table 2)
        "GRF IC": {
            "DPOT": {"l2": (1.64e-1, 7.44e-3), "linf": (4.92e-1, 1.27e-2)},
            "DISCO": {"l2": (6.67e-2, 6.96e-3), "linf": (2.33e-1, 2.47e-2)},
            "HFluxNO": {"l2": (4.14e-2, 1.17e-3), "linf": (1.78e-1, 4.24e-3)},
        },
        "shock IC": {
            "DPOT": {"l2": (1.76e-1, 4.69e-3), "linf": (9.29e-1, 4.45e-2)},
            "DISCO": {"l2": (9.01e-2, 8.08e-3), "linf": (5.39e-1, 2.21e-2)},
            "HFluxNO": {"l2": (6.06e-2, 2.02e-3), "linf": (3.83e-1, 7.62e-3)},
        },
    },
}

In [ ]:
plot_accuracy(results, out="../../figures/fig2.pdf", width=6.0, height=2.5)